# With the obtained best delta_max, do meta learning 
# Feature: metadata of datasets
# Label: obtained best delta_max of datasets
# Multiclass classification

In [ ]:
import os
import pandas as pd
import numpy as np
import io
from scipy.io import arff
from scipy import stats
from scipy.stats import kurtosis, skew
from itertools import combinations
import minepy
from collections import Counter
from sklearn.metrics import mutual_info_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

In [ ]:
import re
############################ Data feature names cleaning ############################
# This function cleans 'thoughts ?' -> 'thoughts?' and handles extra spaces
def clean_column_names(df):
    new_columns = []
    for col in df.columns:
        # 1. Remove leading/trailing whitespace
        c = col.strip()
        # 2. Replace multiple spaces with a single space
        c = re.sub(r'\s+', ' ', c)
        # 3. Remove space before punctuation (e.g., " ?" -> "?")
        c = re.sub(r'\s+([?.!,])', r'\1', c)
        new_columns.append(c)
    
    df.columns = new_columns
    return df

############################ Step 1: Identify Feature Types and Numericalize ############################
def identify_feature_types(df, t=15):
    metadata_list = []
    
    for col in df.columns:
        unique_vals = np.sort(df[col].unique())
        n_unique = len(unique_vals)
        dtype = df[col].dtype
        
        # --- Logic Gates for Categorization ---
        if n_unique <= t:
            # All low-cardinality features are treated as Categorical (Snapping required)
            category = 'Categ'
        
        else:
            if np.issubdtype(dtype, np.integer):
                # High-cardinality integers (Age, Study Hours)
                category = 'Num_disc'
            elif dtype == 'object':
                # High-cardinality strings (Names, IDs, Addresses)
                category = 'REMOVE'
            else:
                category = 'Num_cont' # Fallback for unknown numeric types
        
        metadata_list.append({
            'Feature': col,
            'Num Unique': n_unique,
            'Data Type': dtype,
            'Valid Values': unique_vals.tolist(),
            'Category': category
        })
    metadata = pd.DataFrame(metadata_list)
    Categ = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'Categ']
    Num_disc = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'Num_disc']
    Num_cont = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'Num_cont']
    REMOVE = [metadata.loc[i, 'Feature'] for i in range(len(metadata)) if metadata.loc[i, 'Category'] == 'REMOVE']
        
    return metadata, Categ, Num_disc, Num_cont, REMOVE

def encode_to_numerical(df_org, REMOVE, Categ, Num_disc):
    # 1. Remove features in the REMOVE list
    df_cleaned = df_org.drop(REMOVE, axis=1)

    # 2. Initialize the translation dictionary (The "Rosetta Stone")
    # We will need this for Step 5 and Step 7
    mapping_dict = {}

    df_num = df_cleaned.copy()

    for col in Categ:
        # Get the valid values we stored in metadata
        # We sort them to ensure the mapping is consistent (e.g., 0 is always the same thing)
        unique_labels = sorted(df_num[col].unique())
        if col == Categ[-1]:
            # label = last column (always, Major = 0, Minor = 1)
            unique_labels = [df_num.iloc[:,-1].value_counts().index[0], df_num.iloc[:,-1].value_counts().index[1]]

        # Create the Forward (Text -> Num) and Reverse (Num -> Text) maps
        forward_map = {label: i for i, label in enumerate(unique_labels)}
        reverse_map = {i: label for i, label in enumerate(unique_labels)}

        # Save to our dictionary
        mapping_dict[col] = {
            'forward': forward_map,
            'reverse': reverse_map
        }

        # Apply the transformation
        df_num[col] = df_num[col].map(forward_map)

    # 3. Ensure all columns are numeric types (float or int) for SMOTE
    # This converts any remaining "object" types that might have slipped through
    df_num = df_num.apply(pd.to_numeric)
    
    return mapping_dict, df_num

In [ ]:
rope = 0.01

import bayesiantests as bt
import matplotlib.pyplot as plt

#https://matplotlib.org/stable/gallery/lines_bars_and_markers/horizontal_barchart_distribution.html
def stacked_bar(results, category_names):
    """
    Parameters
    ----------
    results : dict
        A mapping from question labels to a list of answers per category.
        It is assumed all lists contain the same number of entries and that
        it matches the length of *category_names*.
    category_names : list of str
        The category labels.
    """
    labels = list(results.keys())
    data = np.array(list(results.values()))
    data_cum = data.cumsum(axis=1)
    category_colors = plt.colormaps['RdYlGn'](
        np.linspace(0.15, 0.85, data.shape[1]))

    fig, ax = plt.subplots(figsize=(9.2, 5))
    ax.invert_yaxis()
    ax.xaxis.set_visible(False)
    ax.set_xlim(0, np.sum(data, axis=1).max())

    for i, (colname, color) in enumerate(zip(category_names, category_colors)):
        widths = data[:, i]
        starts = data_cum[:, i] - widths
        rects = ax.barh(labels, widths, left=starts, height=0.5,
                        label=colname, color=color)

        r, g, b, _ = color
        text_color = 'white' if r * g * b < 0.5 else 'darkgrey'
        ax.bar_label(rects, label_type='center', color=text_color)
    ax.legend(ncols=len(category_names), bbox_to_anchor=(0, 1),
              loc='upper left', fontsize='small')

    return fig, ax

In [ ]:
def BST(rope, baselines, ours, dfs):
    comp = []
    basewin = []
    draw = []
    ourswin = []
    z = 0
    for i in range(len(ours)):
        for j in range(len(baselines)):
            names = (baselines[j],ours[i])
            comp.append(names)
            X = np.array(dfs[i][[baselines[j],ours[i]]])
            left, within, right = bt.signtest(X, rope=rope, verbose=True, names=names)
            basewin.append(left)
            draw.append(within)
            ourswin.append(right)        
    results = pd.DataFrame(comp, columns = ["Baseline","Ours"])
    results["Basewin_prob"] = basewin
    results["Draw_prob"] = draw
    results["Ourswin_prob"] = ourswin
    return results

# Obatin Meta Features

In [ ]:
data_name = [] # dataset name
sample = [] # number of samples
imb_ratio = [] # imbalance ratio, major:minor = 1:x
feature = [] # number of features
numfeat = [] # number of numerical features
catfeat = [] # number of categorical features
sam_feat = [] # ratio of samples per features
frac_outlier = [] # number of features with outliers/number of features

numfeat_corstat = [] # stats of correlation of numerical features
numfeat_iqrstat = [] # stats of IQR of numerical features
numfeat_FKstat = [] # Statistics on Fisher's Kurtosis of Numeric Attributes
numfeat_PKstat = [] # Statistics on Pearson's Kurtosis of Numeric Attributes
numfeat_skew = [] # Skewness of Numeric Attributes
numfeat_coeff = [] # Coefficient of Variation
numfeat_mic = [] # Maximal Information Coefficient (MIC) between Numeric Attributes
numfeat_mas = [] # Maximum Asymmetry Score (MAS) between Numeric Attributes
numfeat_mev = [] # Maximum Edge Value (MEV) between Numeric Attributes
numfeat_mcn = [] # Minimum Cell Number (MCN) between Numeric Attributes
numfeat_tic = [] # Total Information Coefficient (TIC) between Numeric Attributes
numfeat_ent = [] # Entropy (Empirical) of Numeric Attributes
numfeat_mui = [] # Mutual Information of Numeric Attributes
overlap_f1 = [] # Maximum Fishers' Discriminant Ratio (F1)
overlap_f2 = [] # Volume of the Overlapping Region (F2)
overlap_f3 = [] # Maximum Individual Feature Efficiency (F3)

In [ ]:
####################################### 1. Attributes with outliers #######################################
def attrWithOutliers(ds):
    attrs = 0
    if num_feat:
        for col in num_feat:
            q1 = ds[col].quantile(0.25)
            q3 = ds[col].quantile(0.75)
            iqr = q3 - q1
            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr
            outliers = ds[(ds[col] < lower_bound) | (ds[col] > upper_bound)]
            if not outliers.empty:
                attrs += 1
    return attrs
####################################### 2. Correlation between Numeric Attributes #######################################
def statsCor_NumAttrs(ds):
    if len(num_feat) < 2:
        return {
            "min": 0,
            "max": 0,
            "avg": 0,
            "sd": 0,
            "var": 0,
            **{f"bin{i+1}": 0 for i in range(10)}}
    tmp = ds[num_feat].corr()
    v_cor = tmp.where(np.triu(np.ones(tmp.shape), k=1).astype(bool)).stack().values
    v_cor = [x for x in v_cor if not np.isnan(x)]
    if not v_cor:
        mincor = 0
        maxcor = 0
        avgcor = 0
        sdcor = 0
        varcor = 0
        histcor = [0] * 10
    else:
        mincor = min(v_cor)
        maxcor = max(v_cor)
        avgcor = np.mean(v_cor)
        sdcor = np.std(v_cor)
        varcor = np.var(v_cor)
        if len(v_cor) > 1 and len(set(v_cor)) > 1:
            bins = np.linspace(min(v_cor), max(v_cor), 11)  # 11 bins to get 10 counts
            histcor, _ = np.histogram(v_cor, bins=bins)
        else:
            histcor = [np.nan] * 10
    histcor_dict = {f"bin{i+1}": val for i, val in enumerate(histcor)}
    result = {
        "min": mincor,
        "max": maxcor,
        "avg": avgcor,
        "sd": sdcor,
        "var": varcor,
        **histcor_dict}
    return result
####################################### 3. Interquartile Range (IQR) #######################################
def statsIQR_NumAttrs(ds):
    v_iqr = []
    if num_feat:
        for col in num_feat:
            v_iqr.append(stats.iqr(ds[col]))
    v_iqr = [x for x in v_iqr if not np.isnan(x)]
    if not v_iqr:
        miniqr = 0
        maxiqr = 0
        avgiqr = 0
        sdiqr = 0
        variqr = 0
        histiqr = [0] * 10
    else:
        miniqr = min(v_iqr)
        maxiqr = max(v_iqr)
        avgiqr = np.mean(v_iqr)
        sdiqr = np.std(v_iqr)
        variqr = np.var(v_iqr)
        if len(v_iqr) > 1 and len(set(v_iqr)) > 1:
            bins = np.linspace(min(v_iqr), max(v_iqr), 11)  # 11 bins to get 10 counts
            histiqr, _ = np.histogram(v_iqr, bins=bins)
        else:
            histiqr = [np.nan] * 10
    histiqr_dict = {f"bin{i+1}": val for i, val in enumerate(histiqr)}
    result = {
        "min": miniqr,
        "max": maxiqr,
        "avg": avgiqr,
        "sd": sdiqr,
        "var": variqr,
        **histiqr_dict}
    return result
####################################### 4. FIsher's Kurtosis of Numeric Attributes #######################################
def statsFKur_NumAttrs(ds):
    v_gkur = []
    if num_feat:
        for col in num_feat:
            v_gkur.append(kurtosis(ds[col], fisher=True)) # Fisher's Kurtosis
    v_gkur = [x for x in v_gkur if not np.isnan(x)]
    if not v_gkur:
        mingkur = 0
        maxgkur = 0
        avggkur = 0
        sdgkur = 0
        vargkur = 0
        histgkur = [0] * 10
    else:
        mingkur = min(v_gkur)
        maxgkur = max(v_gkur)
        avggkur = np.mean(v_gkur)
        sdgkur = np.std(v_gkur)
        vargkur = np.var(v_gkur)
        if len(v_gkur) > 1 and len(set(v_gkur)) > 1:
            bins = np.linspace(min(v_gkur), max(v_gkur), 11)  # 11 bins to get 10 counts
            histgkur, _ = np.histogram(v_gkur, bins=bins)
        else:
            histgkur = [np.nan] * 10
    histgkur_dict = {f"bin{i+1}": val for i, val in enumerate(histgkur)}
    result = {
        "min": mingkur,
        "max": maxgkur,
        "avg": avggkur,
        "sd": sdgkur,
        "var": vargkur,
        **histgkur_dict}
    return result
####################################### 5. Pearson's Kurtosis of Numeric Attributes #######################################
def statsPKur_NumAttrs(ds):
    v_pkur = []
    if num_feat:
        for col in num_feat:
            v_pkur.append(kurtosis(ds[col], fisher=False)) # Pearson Kurtosis
    v_pkur = [x for x in v_pkur if not np.isnan(x)]
    if not v_pkur:
        minpkur = 0
        maxpkur = 0
        avgpkur = 0
        sdpkur = 0
        varpkur = 0
        histpkur = [0] * 10
    else:
        minpkur = min(v_pkur)
        maxpkur = max(v_pkur)
        avgpkur = np.mean(v_pkur)
        sdpkur = np.std(v_pkur)
        varpkur = np.var(v_pkur)
        if len(v_pkur) > 1 and len(set(v_pkur)) > 1:
            bins = np.linspace(min(v_pkur), max(v_pkur), 11)  # 11 bins to get 10 counts
            histpkur, _ = np.histogram(v_pkur, bins=bins)
        else:
            histpkur = [np.nan] * 10
    histpkur_dict = {f"bin{i+1}": val for i, val in enumerate(histpkur)}
    result = {
        "min": minpkur,
        "max": maxpkur,
        "avg": avgpkur,
        "sd": sdpkur,
        "var": varpkur,
        **histpkur_dict}
    return result
####################################### 6. Skewness of Numeric Attributes #######################################
def statsSkew_NumAttrs(ds):
    v_skew = []
    if num_feat:
        for col in num_feat:
            v_skew.append(skew(ds[col]))
    v_skew = [x for x in v_skew if not np.isnan(x)]
    if not v_skew:
        minskew = 0
        maxskew = 0
        avgskew = 0
        sdskew = 0
        varskew = 0
        histskew = [0] * 10
    else:
        minskew = min(v_skew)
        maxskew = max(v_skew)
        avgskew = np.mean(v_skew)
        sdskew = np.std(v_skew)
        varskew = np.var(v_skew)
        if len(v_skew) > 1 and len(set(v_skew)) > 1:
            bins = np.linspace(min(v_skew), max(v_skew), 11)  # 11 bins to get 10 counts
            histskew, _ = np.histogram(v_skew, bins=bins)
        else:
            histskew = [np.nan] * 10
    histskew_dict = {f"bin{i+1}": val for i, val in enumerate(histskew)}
    result = {
        "min": minskew,
        "max": maxskew,
        "avg": avgskew,
        "sd": sdskew,
        "var": varskew,
        **histskew_dict}
    return result
####################################### 7. Coefficient of Variation #######################################
def statsCoV_NumAttrs(ds):
    v_cov = []
    if num_feat:
        for col in num_feat:
            mean_val = ds[col].mean()
            if mean_val != 0: # avoid division by zero
                v_cov.append(ds[col].std() / mean_val)
            else:
                v_cov.append(np.nan)
    v_cov = [x for x in v_cov if not np.isnan(x)]
    if not v_cov:
        mincov = 0
        maxcov = 0
        avgcov = 0
        sdcov = 0
        varcov = 0
        histcov = [0] * 10
    else:
        mincov = min(v_cov)
        maxcov = max(v_cov)
        avgcov = np.mean(v_cov)
        sdcov = np.std(v_cov)
        varcov = np.var(v_cov)

        if len(v_cov) > 1 and len(set(v_cov)) > 1:
            bins = np.linspace(min(v_cov), max(v_cov), 11)  # 11 bins to get 10 counts
            histcov, _ = np.histogram(v_cov, bins=bins)
        else:
            histcov = [np.nan] * 10
    histcov_dict = {f"bin{i+1}": val for i, val in enumerate(histcov)}
    result = {
        "min": mincov,
        "max": maxcov,
        "avg": avgcov,
        "sd": sdcov,
        "var": varcov,
        **histcov_dict}
    return result
####### 8. Maximal Information Coefficient (MIC) between Numeric Attributes #######################################
def statsMIC_NumAttrs(ds):
    v_mic = []
    if len(num_feat) > 1:
        for i, j in combinations(num_feat, 2):
            mine = minepy.MINE()
            mine.compute_score(df[i], df[j])
            v_mic.append(mine.mic())
    v_mic = [x for x in v_mic if not np.isnan(x)]
    if len(v_mic) == 0:
        minmic = 0
        maxmic = 0
        avgmic = 0
        sdmic = 0
        varmic = 0
        histmic = [0] * 10
    else:
        minmic = min(v_mic)
        maxmic = max(v_mic)
        avgmic = np.mean(v_mic)
        sdmic = np.std(v_mic)
        varmic = np.var(v_mic)
        if len(v_mic) > 1 and len(set(v_mic)) > 1:
            bins = np.linspace(min(v_mic), max(v_mic), 11)
            histmic, _ = np.histogram(v_mic, bins=bins)
        else:
            histmic = [np.nan] * 10
    histmic_dict = {f"bin{i+1}": val for i, val in enumerate(histmic)}
    return {
        "min": minmic,
        "max": maxmic,
        "avg": avgmic,
        "sd": sdmic,
        "var": varmic,
        **histmic_dict}
####### 9. Maximum Asymmetry Score (MAS) between Numeric Attributes #######################################
def statsMAS_NumAttrs(ds):
    v_mas = []
    if len(num_feat) > 1:
        for i, j in combinations(num_feat, 2):
            mine = minepy.MINE()
            mine.compute_score(df[i], df[j])
            v_mas.append(mine.mas())
    v_mas = [x for x in v_mas if not np.isnan(x)]
    if len(v_mas) == 0:
        minmas = 0
        maxmas = 0
        avgmas = 0
        sdmas = 0
        varmas = 0
        histmas = [0] * 10
    else:
        minmas = min(v_mas)
        maxmas = max(v_mas)
        avgmas = np.mean(v_mas)
        sdmas = np.std(v_mas)
        varmas = np.var(v_mas)
        if len(v_mas) > 1 and len(set(v_mas)) > 1:
            bins = np.linspace(min(v_mas), max(v_mas), 11)
            histmas, _ = np.histogram(v_mas, bins=bins)
        else:
            histmas = [np.nan] * 10
    histmas_dict = {f"bin{i+1}": val for i, val in enumerate(histmas)}
    return {
        "min": minmas,
        "max": maxmas,
        "avg": avgmas,
        "sd": sdmas,
        "var": varmas,
        **histmas_dict}
####### 10. Maximum Edge Value (MEV) between Numeric Attributes #######################################
def statsMEV_NumAttrs(ds):
    v_mev = []
    if len(num_feat) > 1:
        for i, j in combinations(num_feat, 2):
            mine = minepy.MINE()
            mine.compute_score(df[i], df[j])
            v_mev.append(mine.mev())
    v_mev = [x for x in v_mev if not np.isnan(x)]
    if len(v_mev) == 0:
        minmev = 0
        maxmev = 0
        avgmev = 0
        sdmev = 0
        varmev = 0
        histmev = [0] * 10
    else:
        minmev = min(v_mev)
        maxmev = max(v_mev)
        avgmev = np.mean(v_mev)
        sdmev = np.std(v_mev)
        varmev = np.var(v_mev)
        if len(v_mev) > 1 and len(set(v_mev)) > 1:
            bins = np.linspace(min(v_mev), max(v_mev), 11)
            histmev, _ = np.histogram(v_mev, bins=bins)
        else:
            histmev = [np.nan] * 10
    histmev_dict = {f"bin{i+1}": val for i, val in enumerate(histmev)}
    return {
        "min": minmev,
        "max": maxmev,
        "avg": avgmev,
        "sd": sdmev,
        "var": varmev,
        **histmev_dict}
####### 11. Minimum Cell Number (MCN) between Numeric Attributes #######################################
def statsMCN_NumAttrs(ds):
    v_mcn = []
    if len(num_feat) > 1:
        for i, j in combinations(num_feat, 2):
            mine = minepy.MINE()
            mine.compute_score(df[i], df[j])
            v_mcn.append(mine.mcn())
    v_mcn = [x for x in v_mcn if not np.isnan(x)]
    if len(v_mcn) == 0:
        minmcn = 0
        maxmcn = 0
        avgmcn = 0
        sdmcn = 0
        varmcn = 0
        histmcn = [0] * 10
    else:
        minmcn = min(v_mcn)
        maxmcn = max(v_mcn)
        avgmcn = np.mean(v_mcn)
        sdmcn = np.std(v_mcn)
        varmcn = np.var(v_mcn)
        if len(v_mcn) > 1 and len(set(v_mcn)) > 1:
            bins = np.linspace(min(v_mcn), max(v_mcn), 11)
            histmcn, _ = np.histogram(v_mcn, bins=bins)
        else:
            histmcn = [np.nan] * 10
    histmcn_dict = {f"bin{i+1}": val for i, val in enumerate(histmcn)}
    return {
        "min": minmcn,
        "max": maxmcn,
        "avg": avgmcn,
        "sd": sdmcn,
        "var": varmcn,
        **histmcn_dict}
####### 12. Total Information Coefficient (TIC) between Numeric Attributes #######################################
def statsTIC_NumAttrs(ds):
    v_tic = []
    if len(num_feat) > 1:
        for i, j in combinations(num_feat, 2):
            mine = minepy.MINE()
            mine.compute_score(df[i], df[j])
            v_tic.append(mine.tic())
    v_tic = [x for x in v_tic if not np.isnan(x)]
    if len(v_tic) == 0:
        mintic = 0
        maxtic = 0
        avgtic = 0
        sdtic = 0
        vartic = 0
        histtic = [0] * 10
    else:
        mintic = min(v_tic)
        maxtic = max(v_tic)
        avgtic = np.mean(v_tic)
        sdtic = np.std(v_tic)
        vartic = np.var(v_tic)
        if len(v_tic) > 1 and len(set(v_tic)) > 1:
            bins = np.linspace(min(v_tic), max(v_tic), 11)
            histtic, _ = np.histogram(v_tic, bins=bins)
        else:
            histtic = [np.nan] * 10
    histtic_dict = {f"bin{i+1}": val for i, val in enumerate(histtic)}
    return {
        "min": mintic,
        "max": maxtic,
        "avg": avgtic,
        "sd": sdtic,
        "var": vartic,
        **histtic_dict}
####################################### 13. Entropy (Empirical) of Attributes #######################################
def statsEnt(ds):
    v_ent = []
    if num_feat:
        for col in num_feat:
            if pd.api.types.is_numeric_dtype(ds[col]):
                counts = Counter(ds[col].dropna())
                probs = [float(c) / len(ds[col].dropna()) for c in counts.values()]
                entropy_val = stats.entropy(probs, base=2)
                v_ent.append(entropy_val)
            else:
                counts = Counter(ds[col].dropna())
                probs = [float(c) / len(ds[col].dropna()) for c in counts.values()]
                entropy_val = stats.entropy(probs, base=2)
                v_ent.append(entropy_val)
    v_ent = [x for x in v_ent if not np.isnan(x)]
    if not v_ent:
        minent = 0
        maxent = 0
        avgent = 0
        sdent = 0
        varent = 0
        histent = [0] * 10
    else:
        minent = min(v_ent)
        maxent = max(v_ent)
        avgent = np.mean(v_ent)
        sdent = np.std(v_ent)
        varent = np.var(v_ent)
        if len(v_ent) > 1 and len(set(v_ent)) > 1:
            bins = np.linspace(min(v_ent), max(v_ent), 11)
            histent, _ = np.histogram(v_ent, bins=bins)
        else:
            histent = [np.nan] * 10
    histent_dict = {f"bin{i+1}": val for i, val in enumerate(histent)}
    result = {
        "min": minent,
        "max": maxent,
        "avg": avgent,
        "sd": sdent,
        "var": varent,
        **histent_dict}
    return result
####################################### 14. Mutual Information of Attributes #######################################
def statsMuI(ds):
    if not num_feat:
        return {
            "min": 0,
            "max": 0,
            "avg": 0,
            "sd": 0,
            "var": 0,
            **{f"bin{i+1}": 0 for i in range(10)}}
    tmpM = ds[num_feat].apply(lambda x: pd.qcut(x, q=10, labels=False, duplicates='drop'))
    tmp_res = tmpM.apply(lambda col1: tmpM.apply(lambda col2: mutual_info_score(col1.dropna(), col2.dropna())))
    v_mui = tmp_res.where(np.triu(np.ones(tmp_res.shape), k=1).astype(bool)).stack().values
    v_mui = [x for x in v_mui if not np.isnan(x)]
    if not v_mui:
        minmui = 0
        maxmui = 0
        avgmui = 0
        sdmui = 0
        varmui = 0
        histmui = [0] * 10
    else:
        minmui = min(v_mui)
        maxmui = max(v_mui)
        avgmui = np.mean(v_mui)
        sdmui = np.std(v_mui)
        varmui = np.var(v_mui)
        if len(v_mui) > 1 and len(set(v_mui)) > 1:
            bins = np.linspace(min(v_mui), max(v_mui), 11)
            histmui, _ = np.histogram(v_mui, bins=bins)
        else:
            histmui = [np.nan] * 10
    histmui_dict = {f"bin{i+1}": val for i, val in enumerate(histmui)}
    result = {
        "min": minmui,
        "max": maxmui,
        "avg": avgmui,
        "sd": sdmui,
        "var": varmui,
        **histmui_dict}
    return result
####################################### 15. Measures of Overlapping #######################################
def calculate_f1(X, y):
    """Calculates F1 (Maximum Fisher's Discriminant Ratio) robustly."""
    # Ensure inputs are NumPy arrays
    X = np.asarray(X)
    y = np.asarray(y)
    
    # 1. Remove features with zero variance (constant columns)
    # This prevents many 'Singular Matrix' errors before they start
    non_constant_mask = np.var(X, axis=0) > 0
    X = X[:, non_constant_mask]
    
    if X.shape[1] == 0:
        return 0.0
    
    n_features = X.shape[1]
    mean_overall = X.mean(axis=0)
    classes = np.unique(y)
    
    Sb = np.zeros((n_features, n_features))
    Sw = np.zeros((n_features, n_features))
    
    for c in classes:
        X_c = X[y == c]
        if len(X_c) < 2: 
            continue 
        
        mean_c = X_c.mean(axis=0)
        # Between-class scatter
        diff = (mean_c - mean_overall).reshape(-1, 1)
        Sb += len(X_c) * (diff @ diff.T)
        
        # Within-class scatter (using degrees of freedom)
        Sw += np.cov(X_c.T, ddof=1) * (len(X_c) - 1)
    
    # 2. Add regularization (shrinkage) to ensure Sw is invertible
    # Even a small epsilon (1e-6) makes the math stable
    epsilon = 1e-6
    Sw += np.eye(n_features) * epsilon
    
    # 3. Use pinv (pseudo-inverse) for maximum safety against singularity
    try:
        # Solving the generalized eigenvalue problem
        S = np.linalg.pinv(Sw).dot(Sb)
        eigenvalues = np.linalg.eigvals(S)
        # Return max real part (eigenvalues can be complex due to precision)
        return float(np.max(eigenvalues.real))
    except Exception:
        return 0.0

def calculate_f2(X, y):
    """Calculates F2 (Volume of the Overlapping Region) - Simplified Approximation."""
    classes = np.unique(y)
    ranges = {}
    for c in classes:
        ranges[c] = [(X[y == c][col].min(), X[y == c][col].max()) for col in X.columns]

    overlap_volume = 1.0
    for col in X.columns:
        overlaps = []
        for c1, c2 in combinations(classes, 2):
            min_overlap = max(ranges[c1][X.columns.get_loc(col)][0], ranges[c2][X.columns.get_loc(col)][0])
            max_overlap = min(ranges[c1][X.columns.get_loc(col)][1], ranges[c2][X.columns.get_loc(col)][1])
            if max_overlap > min_overlap:
                overlaps.append(max_overlap - min_overlap)
        if overlaps:
            overlap_volume *= sum(overlaps) / (len(list(combinations(classes, 2))) * (X[col].max() - X[col].min())) if (X[col].max() - X[col].min()) > 0 else 0
        else:
            overlap_volume*=0

    return overlap_volume

def calculate_f3(X, y):
    """Calculates F3 (Maximum Individual Feature Efficiency)."""
    classes = np.unique(y)
    efficiencies = []
    
    for col in X.columns:
        class_variances = []
        for c in classes:
            X_c = X[y == c][col]
            class_variances.append(X_c.var())
        
        # Feature efficiency: ratio of between-class variance to within-class variance
        between_class_variance = X[col].groupby(y).mean().var()
        within_class_variance = np.mean(class_variances)
        efficiencies.append(between_class_variance / within_class_variance if within_class_variance > 0 else 0)
        
    return np.max(efficiencies)

In [ ]:
for i in range(1, 98):
    start = time.time()
    if i == 23 or i == 81 or i == 82 or i == 84:        # duplicated
        continue
    # Load data
    df = pd.read_csv(r'/5. SMOTE_LLM/data_original/'+'ds'+ str(i) +'.csv')
    df = clean_column_names(df)                                                         # Clean Feature Names
    metadata, Categ, Num_disc, Num_cont, REMOVE = identify_feature_types(df, t=15)      # Identifying Feature Types
    mapping_dict, df = encode_to_numerical(df, REMOVE, Categ, Num_disc)                 # Numericalizing the dataset
    print('+'*35, '{} Dataset'.format(i), '+'*35)
    print('<Modified Class>\n', df.iloc[:,-1].value_counts())
    print('<Imabalance ratio>\n', "1:{: .2f}".format(df.iloc[:,-1].value_counts()[1]/df.iloc[:,-1].value_counts()[0]))
    data_name.append(i) # dataset name
    ################################################################################# 
    # Distinguish [Cat, Num_cont, Num_disc] and Numericalize the data
    metadata, Categ, Num_disc, Num_cont, REMOVE = identify_feature_types(df, t=15)
    mapping_dict, df = encode_to_numerical(df, REMOVE, Categ, Num_disc)  
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]
    #################################################################################
    sample.append(X.shape[0]) # number of samples
    feature.append(X.shape[1]) # number of features
    imb_ratio.append(y.value_counts()[0]/y.value_counts()[1]) # imbalance ratio, major:minor = x:1
    num_feat = Num_disc + Num_cont
    cat_feat = Categ
    numfeat.append(len(num_feat)) # number of numerical features
    catfeat.append(len(cat_feat)-1) # number of categorical features, exclude Label
    sam_feat.append(X.shape[0]/X.shape[1]) # ratio of samples per features
    ######################################
    # 1. Attributes with outliers
    attrs = attrWithOutliers(df)
    frac_outlier.append(attrs/X.shape[1])
    ######################################
    # 2. Correlation between Numeric Attributes
    result = statsCor_NumAttrs(df)
    numfeat_corstat.append(result)
    ######################################
    # 3. Interquartile Range (IQR) 
    result = statsIQR_NumAttrs(df)
    numfeat_iqrstat.append(result)
    ######################################
    # 4. FIsher's Kurtosis of Numeric Attributes
    result = statsFKur_NumAttrs(df)
    numfeat_FKstat.append(result)
    ######################################
    # 5. Pearson's Kurtosis of Numeric Attributes
    result = statsPKur_NumAttrs(df)
    numfeat_PKstat.append(result)
    ######################################
    # 6. Skewness of Numeric Attributes
    result = statsSkew_NumAttrs(df)
    numfeat_skew.append(result)
    ######################################
    # 7. Coefficient of Variation
    result = statsCoV_NumAttrs(df)
    numfeat_coeff.append(result)
    ######################################
    # 8. Maximal Information Coefficient (MIC) between Numeric Attributes
    result = statsMIC_NumAttrs(df)
    numfeat_mic.append(result)
    ######################################
    # 9. Maximum Asymmetry Score (MAS) between Numeric Attributes
    result = statsMAS_NumAttrs(df)
    numfeat_mas.append(result)
    ######################################
    # 10. Maximum Edge Value (MEV) between Numeric Attributes
    result = statsMEV_NumAttrs(df)
    numfeat_mev.append(result)
    ######################################
    # 11. Minimum Cell Number (MCN) between Numeric Attributes
    result = statsMCN_NumAttrs(df)
    numfeat_mcn.append(result)
    ######################################
    # 12. Total Information Coefficient (TIC) between Numeric Attributes
    result = statsTIC_NumAttrs(df)
    numfeat_tic.append(result)
    ######################################
    # 13. Entropy (Empirical) of Attributes
    result = statsEnt(df)
    numfeat_ent.append(result)
    ######################################
    # 14. Mutual Information of Attributes
    result = statsMuI(df)
    numfeat_mui.append(result)
    ######################################
    # 15. Measures of Overlapping
    result = calculate_f1(X, y)
    overlap_f1.append(result)    # not calculated with some datasets
    result = calculate_f2(X, y)
    overlap_f2.append(result)
    result = calculate_f3(X, y)
    overlap_f3.append(result)
print("complete")

In [ ]:
df_meta = pd.DataFrame(data_name, columns=["data"] )
df_meta["sample#"] = sample
df_meta["feature#"] = feature
df_meta["Imb_Ratio(x:1)"] = imb_ratio
df_meta["numerical"] = numfeat
df_meta["categorical"] = catfeat
df_meta["samp/feat"] = sam_feat
df_meta['num/feat'] = df_meta.loc[:,"numerical"]/df_meta.loc[:,"feature#"]
df_meta['cat/feat'] = df_meta.loc[:,"categorical"]/df_meta.loc[:,"feature#"]
df_meta["frac_outlier"] = frac_outlier

# stats (main, max, avg, std, var, hist*10), but we exclude hist*10, so total 5 for each
keys = list(numfeat_corstat[0].keys())
# df_meta["numfeat_cor"] = numfeat_corstat
for i in range(len(df_meta)):   # number of datasets
    for j in range(5):  # number of metrics (min, max, ...) = originally 'len(keys)', but we exclude hist*10
        df_meta.loc[i,'cor_'+f'{keys[j]}'] = numfeat_corstat[i][keys[j]]            
# # df_meta["numfeat_iqr"] = numfeat_iqrstat
for i in range(len(df_meta)):   # number of datasets
    for j in range(5):  # number of metrics (min, max, ...) = originally 'len(keys)', but we exclude hist*10
        df_meta.loc[i,'iqr_'+f'{keys[j]}'] = numfeat_iqrstat[i][keys[j]] 
# # df_meta["numfeat_Fisher"] = numfeat_FKstat
for i in range(len(df_meta)):   # number of datasets
    for j in range(5):  # number of metrics (min, max, ...) = originally 'len(keys)', but we exclude hist*10
        df_meta.loc[i,'FK_'+f'{keys[j]}'] = numfeat_FKstat[i][keys[j]] 
# # df_meta["numfeat_Pearson"] = numfeat_PKstat
for i in range(len(df_meta)):   # number of datasets
    for j in range(5):  # number of metrics (min, max, ...) = originally 'len(keys)', but we exclude hist*10
        df_meta.loc[i,'PK_'+f'{keys[j]}'] = numfeat_PKstat[i][keys[j]] 
# # df_meta["numfeat_skew"] = numfeat_skew
for i in range(len(df_meta)):   # number of datasets
    for j in range(5):  # number of metrics (min, max, ...) = originally 'len(keys)', but we exclude hist*10
        df_meta.loc[i,'skew_'+f'{keys[j]}'] = numfeat_skew[i][keys[j]] 
# # df_meta["numfeat_coeff"] = numfeat_coeff
for i in range(len(df_meta)):   # number of datasets
    for j in range(5):  # number of metrics (min, max, ...) = originally 'len(keys)', but we exclude hist*10
        df_meta.loc[i,'coeff_'+f'{keys[j]}'] = numfeat_coeff[i][keys[j]] 
# # df_meta["numfeat_mic"] = numfeat_mic
for i in range(len(df_meta)):   # number of datasets
    for j in range(5):  # number of metrics (min, max, ...) = originally 'len(keys)', but we exclude hist*10
        df_meta.loc[i,'mic_'+f'{keys[j]}'] = numfeat_mic[i][keys[j]] 
# # df_meta["numfeat_mas"] = numfeat_mas
for i in range(len(df_meta)):   # number of datasets
    for j in range(5):  # number of metrics (min, max, ...) = originally 'len(keys)', but we exclude hist*10
        df_meta.loc[i,'mas_'+f'{keys[j]}'] = numfeat_mas[i][keys[j]] 
# # df_meta["numfeat_mev"] = numfeat_mev
for i in range(len(df_meta)):   # number of datasets
    for j in range(5):  # number of metrics (min, max, ...) = originally 'len(keys)', but we exclude hist*10
        df_meta.loc[i,'mev_'+f'{keys[j]}'] = numfeat_mev[i][keys[j]] 
# # df_meta["numfeat_mcn"] = numfeat_mcn
for i in range(len(df_meta)):   # number of datasets
    for j in range(5):  # number of metrics (min, max, ...) = originally 'len(keys)', but we exclude hist*10
        df_meta.loc[i,'mcn_'+f'{keys[j]}'] = numfeat_mcn[i][keys[j]] 
# # df_meta["numfeat_tic"] = numfeat_tic
for i in range(len(df_meta)):   # number of datasets
    for j in range(5):  # number of metrics (min, max, ...) = originally 'len(keys)', but we exclude hist*10
        df_meta.loc[i,'tic_'+f'{keys[j]}'] = numfeat_tic[i][keys[j]] 
# # df_meta["numfeat_ent"] = numfeat_ent
for i in range(len(df_meta)):   # number of datasets
    for j in range(5):  # number of metrics (min, max, ...) = originally 'len(keys)', but we exclude hist*10
        df_meta.loc[i,'ent_'+f'{keys[j]}'] = numfeat_ent[i][keys[j]] 
# # df_meta["numfeat_mui"] = numfeat_mui
for i in range(len(df_meta)):   # number of datasets
    for j in range(5):  # number of metrics (min, max, ...) = originally 'len(keys)', but we exclude hist*10
        df_meta.loc[i,'mui_'+f'{keys[j]}'] = numfeat_mui[i][keys[j]] 
            
# single value again
df_meta["overlap_f1"] = overlap_f1    # not calculated with some datasets
df_meta["overlap_f2"] = overlap_f2
df_meta["overlap_f3"] = overlap_f3

df_meta.to_csv("Meta Results/metadata.csv")

In [ ]:
df_meta = pd.read_csv("Meta Results/metadata.csv", index_col='Unnamed: 0')
df_meta

# With 4 Delta_max values

In [ ]:
# Remove data numbers
df_meta = df_meta.iloc[:,1:]
# labels based on F1
best_delta = pd.read_csv("Meta Results/Best_Deltas(test).csv")
best_delta_f1 = best_delta.iloc[:,1]

# Meta Learning

## 1) Prepare Meta Dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# 1. Basic checks
print("df_meta shape:", df_meta.shape)
print("best_delta length:", len(best_delta_f1))

# Convert to numpy
y_raw = np.array(best_delta_f1)

# IMPORTANT: treat delta as categorical
le = LabelEncoder()
y = le.fit_transform(y_raw)

print("Original classes:", le.classes_)
print("Encoded labels:", np.unique(y))

# 2. Check alignment
assert df_meta.shape[0] == len(y), "Mismatch between features and labels!"

# 3. Inspect target distribution
unique, counts = np.unique(y, return_counts=True)
print("\nDelta distribution:")
for u, c in zip(unique, counts):
    print(f"delta={u}: {c}")

# 4. Check missing values
print("\nMissing values per column:")
missing_counts = df_meta.isnull().sum()
print(missing_counts[missing_counts > 0])

# 5. Fill missing values (simple and safe)
df_meta_filled = df_meta.fillna(df_meta.median(numeric_only=True))

# 6. Convert to numpy
X = df_meta_filled.values

print("\nFinal X shape:", X.shape)
print("Final y shape:", y.shape)

## 2) Implement Leave-One-Out CV (LOOCV)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score

# LOOCV
loo = LeaveOneOut()

y_true = []
y_pred = []

for train_idx, test_idx in loo.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Model (simple version first)
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )

    # Train
    model.fit(X_train, y_train)

    # Predict
    pred = model.predict(X_test)[0]

    # Store
    y_true.append(y_test[0])
    y_pred.append(pred)

# Convert to arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Basic evaluation
acc = accuracy_score(y_true, y_pred)

print("LOOCV Matching Accuracy:", acc)

# Show predictions vs truth
print("\nSample predictions (first 20):")
for i in range(min(20, len(y_true))):
    print(f"True: {y_true[i]}, Pred: {y_pred[i]}")

In [ ]:
print("\nPredicted distribution:")
unique, counts = np.unique(y_pred, return_counts=True)
for u, c in zip(unique, counts):
    print(f"delta={u}: {c}")

## 3) Performance Gap Evaluation
##  *True: Real best value in the test scores
##  *Pred: predicted delta_max from trained meta_learner (above)

In [ ]:
df_test_f1 = pd.read_csv("Meta Results/test_scores_from_F1.csv")
df_test_roc = pd.read_csv("Meta Results/test_scores_from_ROCAUC.csv")
df_test_pr = pd.read_csv("Meta Results/test_scores_from_PRAUC.csv")
df_test_f1

In [ ]:
deltas = ['1.5', '2.0', '2.5', '3.0']

In [ ]:
# performance gap 
score_list = []
for i, row in df_test_f1.iterrows():
    score_pred = row[y_pred[i]+1]
    score_real = row[np.argmax(row[1:-1])+1]
    print(score_real, score_pred, float(score_real)-float(score_pred))
    score_list.append(float(score_real)-float(score_pred))
print(np.average(score_list), np.std(score_list)) 

In [ ]:
# Number of Matched
score_list.count(0)/len(score_list)

In [ ]:
# rank
df_test_rank = df_test_f1.iloc[:,1:5].rank(axis=1, ascending=False, method='min')
df_test_rank

In [ ]:
rank_list = []
for i, row in df_test_rank.iterrows():
    rank_num = df_test_rank.iloc[i, y_pred[i]]
    rank_list.append(rank_num)
print(np.mean(rank_list), np.std(rank_list))
print(rank_list)

In [ ]:
avg_list = []
for i, row in df_test_rank.iterrows():
    rank_avg = np.mean(row)
    avg_list.append(rank_avg)
print(np.mean(avg_list), np.std(avg_list))
print(avg_list)

In [ ]:
rank_comp = pd.DataFrame(rank_list, columns=['pred_rank'])
rank_comp['avg_rank'] = avg_list
rank_comp

In [ ]:
# Average
print("==========", "Average", "==========")
print("Pred:",np.mean(rank_comp.iloc[:,0]))
print("Avg:",np.mean(rank_comp.iloc[:,1]))

# Bayesian Sign Test
# This test shows that the probabilities (Win/Draw/Lose) based on the results (the higher is winning)
# the higher is winning, so win prob means that the probability of being higher

print("============"*3, "Probability", "============"*3)
baselines = ['avg_rank']
ours = ['pred_rank']
dfs = [rank_comp]

comp = []
basewin = []
draw = []
ourswin = []
z = 0
for i in range(len(ours)):
    for j in range(len(baselines)):
#         print(z)
        names = (baselines[j],ours[i])
#         print(names)
        comp.append(names)
        X_bst = np.array(dfs[i][[baselines[j],ours[i]]])
        left, within, right = bt.signtest(X_bst, rope=rope, verbose=True, names=names)
#         print(left, within, right)
        basewin.append(left)
        draw.append(within)
        ourswin.append(right)
        
results = pd.DataFrame(comp, columns = ["Baseline","Ours"])
results["Basewin_prob"] = basewin
results["Draw_prob"] = draw
results["Ourswin_prob"] = ourswin
results

## 4) feature reduction (optional), if we do feature reduction, need to do step 2&3 again

In [ ]:
# Feature importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
import numpy as np

loo = LeaveOneOut()

feature_importances = []

for train_idx, test_idx in loo.split(X):
    X_train, y_train = X[train_idx], y[train_idx]

    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )

    model.fit(X_train, y_train)
    feature_importances.append(model.feature_importances_)

# average importance across folds
mean_importance = np.mean(feature_importances, axis=0)

# sort features
indices = np.argsort(mean_importance)[::-1]

# print top features
print("Top 20 features:")
for i in range(20):
    print(f"{i+1}. Feature {indices[i]} - Importance: {mean_importance[indices[i]]:.4f}")

## 1. Main Meta Learner Analysis

In [ ]:
# With all features (without feature reduction)
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
metrics = ['F1', 'GM', 'BA', 'ROCAUC', 'PRAUC']

for i in range(5):
    if i == 1 or i == 2:
        continue
    print('*'*100)
    print(f"Results for {metrics[i]}")
    # Training with current ground truth (from CV results)
    best_delta = pd.read_csv("Meta Results/Best_Deltas(test).csv")
    best_delta = best_delta.iloc[:,i+1]
    # Convert to numpy
    y_raw = np.array(best_delta)
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    unique, counts = np.unique(y, return_counts=True)
    print("Ground Truth Delta distribution:")
    for u, c in zip(unique, counts):
        print(f"delta={u}: {c}")
    # LODO-CV
    loo = LeaveOneOut()
    y_true = []
    y_pred = []
    for train_idx, test_idx in loo.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        # Model (simple version first)
        model = RandomForestClassifier(
            n_estimators=100,
            random_state=42)
        # Train
        model.fit(X_train, y_train)
        # Predict
        pred = model.predict(X_test)[0]
        # Store
        y_true.append(y_test[0])
        y_pred.append(pred)
    # Convert to arrays
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    # Basic evaluation
    acc = accuracy_score(y_true, y_pred)
    print("LODO-CV Matching Accuracy:", acc)
    print("Predicted distribution:")
    unique, counts = np.unique(y_pred, return_counts=True)
    for u, c in zip(unique, counts):
        print(f"delta={u}: {c}")
    
    # performance gap / Matching Accuracy
    df_test = pd.read_csv(f"Meta Results/test_scores_{metrics[i]}.csv")
    unique, counts = np.unique(df_test.iloc[:,-1], return_counts=True)
    print("Real Results Delta distribution:")
    for u, c in zip(unique, counts):
        print(f"delta={u}: {c}")
    score_list = []
    for i, row in df_test.iterrows():
        score_pred = row[y_pred[i]+1]
        score_real = row[np.argmax(row[1:-1])+1]
#         print(score_real, score_pred, float(score_real)-float(score_pred))
        score_list.append(float(score_real)-float(score_pred))
    print("Performance Gap (Mean, Std):", np.average(score_list), ",", np.std(score_list))
    print("Matching Accuracy:", score_list.count(0)/len(score_list)) 
    
    # Ranking
    df_test_rank = df_test.iloc[:,1:5].rank(axis=1, ascending=False, method='min')
    rank_list = []
    for i, row in df_test_rank.iterrows():
        rank_num = df_test_rank.iloc[i, y_pred[i]]
        rank_list.append(rank_num)
    print('Average Rank:', np.average(rank_list))
    
    # Ranking - BST Analysis
    rank_comp = pd.DataFrame(rank_list, columns=['pred_rank'])
    rank_comp['avg_rank'] = avg_list

    print("==========", "Average Ranking", "==========")
    print("Pred:",np.mean(rank_comp.iloc[:,0]))
    print("Avg:",np.mean(rank_comp.iloc[:,1]))

    print("============"*3, "Probability", "============"*3)
    baselines = ['avg_rank']
    ours = ['pred_rank']
    dfs = [rank_comp]

    comp = []
    basewin = []
    draw = []
    ourswin = []
    z = 0
    for i in range(len(ours)):
        for j in range(len(baselines)):
            names = (baselines[j],ours[i])
            comp.append(names)
            X_bst = np.array(dfs[i][[baselines[j],ours[i]]])
            left, within, right = bt.signtest(X_bst, rope=rope, verbose=True, names=names)
            basewin.append(left)
            draw.append(within)
            ourswin.append(right)

    results = pd.DataFrame(comp, columns = ["Baseline","Ours"])
    results["Basewin_prob"] = basewin
    results["Draw_prob"] = draw
    results["Ourswin_prob"] = ourswin
    results

In [ ]:
import joblib
print(X.shape, y.shape)
model = RandomForestClassifier(n_estimators=100,random_state=42)
model.fit(X, y)
# save model
joblib.dump(model, 'Meta Results/rf_77.pkl')

## Reduced features

In [ ]:
# sorted indices from previous step
top_indices = indices

# choose K
K = 20   # change to 30, 40 later

selected_features = top_indices[:K]

X_reduced = X[:, selected_features]

print("Reduced shape:", X_reduced.shape)

In [ ]:
# top 20
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
metrics = ['F1', 'GM', 'BA', 'ROCAUC', 'PRAUC']

for i in range(5):
    if i == 1 or i == 2:
        continue
    print('*'*100)
    print(f"Results for {metrics[i]}")
    # Training with current ground truth (from CV results)
    best_delta = pd.read_csv("Meta Results/Best_Deltas(test)_from.csv")
    best_delta = best_delta.iloc[:,i+1]
    # Convert to numpy
    y_raw = np.array(best_delta)
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    unique, counts = np.unique(y, return_counts=True)
    print("Ground Truth Delta distribution:")
    for u, c in zip(unique, counts):
        print(f"delta={u}: {c}")
    # LODO-CV
    loo = LeaveOneOut()
    y_true = []
    y_pred = []
    for train_idx, test_idx in loo.split(X_reduced):
        X_train, X_test = X_reduced[train_idx], X_reduced[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        # Model (simple version first)
        model = RandomForestClassifier(
            n_estimators=100,
            random_state=42)
        # Train
        model.fit(X_train, y_train)
        # Predict
        pred = model.predict(X_test)[0]
        # Store
        y_true.append(y_test[0])
        y_pred.append(pred)
    # Convert to arrays
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    # Basic evaluation
    acc = accuracy_score(y_true, y_pred)
    print("LODO-CV Matching Accuracy:", acc)
    print("Predicted distribution:")
    unique, counts = np.unique(y_pred, return_counts=True)
    for u, c in zip(unique, counts):
        print(f"delta={u}: {c}")
    
    # performance gap / Matching Accuracy
    df_test = pd.read_csv(f"Meta Results/test_scores_from_{metrics[i]}.csv")
    unique, counts = np.unique(df_test.iloc[:,-1], return_counts=True)
    print("Real Results Delta distribution:")
    for u, c in zip(unique, counts):
        print(f"delta={u}: {c}")
    score_list = []
    for i, row in df_test.iterrows():
        score_pred = row[y_pred[i]+1]
        score_real = row[np.argmax(row[1:-1])+1]
#         print(score_real, score_pred, float(score_real)-float(score_pred))
        score_list.append(float(score_real)-float(score_pred))
    print("Performance Gap (Mean, Std):", np.average(score_list), ",", np.std(score_list))
    print("Matching Accuracy:", score_list.count(0)/len(score_list)) 
    
    # Ranking
    df_test_rank = df_test.iloc[:,1:5].rank(axis=1, ascending=False, method='min')
    rank_list = []
    for i, row in df_test_rank.iterrows():
        rank_num = df_test_rank.iloc[i, y_pred[i]]
        rank_list.append(rank_num)
    print('Average Rank:', np.average(rank_list))
    
    # Ranking - BST Analysis
    rank_comp = pd.DataFrame(rank_list, columns=['pred_rank'])
    rank_comp['avg_rank'] = avg_list

    print("==========", "Average Ranking", "==========")
    print("Pred:",np.mean(rank_comp.iloc[:,0]))
    print("Avg:",np.mean(rank_comp.iloc[:,1]))

    print("============"*3, "Probability", "============"*3)
    baselines = ['avg_rank']
    ours = ['pred_rank']
    dfs = [rank_comp]

    comp = []
    basewin = []
    draw = []
    ourswin = []
    z = 0
    for i in range(len(ours)):
        for j in range(len(baselines)):
            names = (baselines[j],ours[i])
            comp.append(names)
            X_bst = np.array(dfs[i][[baselines[j],ours[i]]])
            left, within, right = bt.signtest(X_bst, rope=rope, verbose=True, names=names)
            basewin.append(left)
            draw.append(within)
            ourswin.append(right)

    results = pd.DataFrame(comp, columns = ["Baseline","Ours"])
    results["Basewin_prob"] = basewin
    results["Draw_prob"] = draw
    results["Ourswin_prob"] = ourswin
    results

In [ ]:
import joblib
print(X_reduced.shape, y.shape)
model = RandomForestClassifier(n_estimators=100,random_state=42)
model.fit(X_reduced, y)
joblib.dump(model, 'Meta Results/rf_20.pkl')

## 2. Comparison with Baselines (Fixed Delta Choice)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
metrics = ['F1', 'GM', 'BA', 'ROCAUC', 'PRAUC']

for i in range(5):
    if i == 1 or i == 2:
        continue
    print(f"!!!Results for {metrics[i]}!!!")

    # Generating Random Index
    import random
    # random.seed(42)
    # Change the fixed del_max
    y_pred = [0] * len(df_test)
#     y_pred = [1] * len(df_test)
#     y_pred = [2] * len(df_test)
#     y_pred = [3] * len(df_test)
    
    # performance gap / Matching Accuracy
    df_test = pd.read_csv(f"Meta Results/test_scores_from_{metrics[i]}.csv")
    unique, counts = np.unique(df_test.iloc[:,-1], return_counts=True)
    print("Real Results Delta distribution:")
    for u, c in zip(unique, counts):
        print(f"delta={u}: {c}")
    score_list = []
    for i, row in df_test.iterrows():
        score_pred = row[y_pred[i]+1]
#         print(score_pred)
        score_real = row[np.argmax(row[1:-1])+1]
#         print(score_real, score_pred, float(score_real)-float(score_pred))
        score_list.append(float(score_real)-float(score_pred))
    print("Performance Gap (Mean, Std):", np.average(score_list), ",", np.std(score_list))
    print("Matching Accuracy:", score_list.count(0)/len(score_list)) 
    
    # Ranking
    df_test_rank = df_test.iloc[:,1:5].rank(axis=1, ascending=False, method='min')
    rank_list = []
    for i, row in df_test_rank.iterrows():
        rank_num = df_test_rank.iloc[i, y_pred[i]]
        rank_list.append(rank_num)
    print('Average Rank:', np.average(rank_list))
    
    # Ranking - BST Analysis
    rank_comp = pd.DataFrame(rank_list, columns=['random_rank'])
    rank_comp['avg_rank'] = avg_list

    print("==========", "Average Ranking", "==========")
    print("Random:",np.mean(rank_comp.iloc[:,0]))
    print("Avg:",np.mean(rank_comp.iloc[:,1]))

    print("============"*3, "Probability", "============"*3)
    baselines = ['avg_rank']
    ours = ['random_rank']
    dfs = [rank_comp]

    comp = []
    basewin = []
    draw = []
    ourswin = []
    z = 0
    for i in range(len(ours)):
        for j in range(len(baselines)):
            names = (baselines[j],ours[i])
            comp.append(names)
            X_bst = np.array(dfs[i][[baselines[j],ours[i]]])
            left, within, right = bt.signtest(X_bst, rope=rope, verbose=True, names=names)
            basewin.append(left)
            draw.append(within)
            ourswin.append(right)

    results = pd.DataFrame(comp, columns = ["Baseline","Ours"])
    results["Basewin_prob"] = basewin
    results["Draw_prob"] = draw
    results["Ourswin_prob"] = ourswin
    results
    